1. Load tt_df
2. Create a dataset of 1000 transcripts 
3. Load yt_df

In [10]:
import os
import json

import pandas as pd
from tqdm import tqdm 
import pickle
import pprint

from fuzzywuzzy import fuzz as fw
import numpy as np

import matplotlib.pyplot as plt

if os.path.isdir("../data"):
    os.chdir("../")

%load_ext autoreload
%autoreload 2

from src.transcript_matching import *



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### helper function

In [11]:
def load_transcript(videoid):
    try:
        with open(f'data/test_transcript_collection_accuracy/transcripts/{videoid}.json') as f:
            transcript = json.load(f)
        return transcript['text']
    except FileNotFoundError as e:
        return None

# 1. Transcript collection

## 1.1 TikTok

### 1.1.1 Create test file with video_id for which I already have the transcripts from tiktok api

```python
# 1. Create a DF which contains 2 samples of tiktok w transcripts from each channel and each file
tiktok_w_transcripts = []
for year in ["2022", "2024"]:
    for channel_type in ["pp", "news"]:
        df = load_transcripts_only_df(year=year, channel_type=channel_type, platform="tiktok")
        df = pd.DataFrame(df)
        df = df[(~df['transcript'].isna()) & (df['transcript']!= '')]
        channels = df.channel.unique()
        
        print(f"{channel_type}, {year}, {len(channels)}")
        for channel in channels:
            df_channel = df[df['channel']==channel]
            size_sample = min(2, len(df_channel))
            tiktok_w_transcripts.append(df_channel.sample(size_sample, random_state=0))
tiktok_w_transcripts = pd.concat(tiktok_w_transcripts)
tiktok_w_transcripts.to_json("data/test_transcript_collection_accuracy/tiktok_test.jsonl", lines=True, orient="records")
```

tiktok_w_transcripts:

| index | video_id | channel | transcript |
|---|---|---|---|
| 1 | 7071953261699370246 | zemmour_eric | Et là c'est la technique pour être bon. C'est ... |
| 6 | 7065345831892929797 | zemmour_eric | mais aujourd'hui le courage c'est vous qui ave... |
| 19 | 7064258884084567301 | jordanbardella | il y a des centaines et des centaines de quart... |
| 24 | 7093916115218222342 | jordanbardella | Il y a des millions de Français qui sont plong... |
| 48 | 7090096316516240645 | mlp.officiel | Madame Le Pen, par ici. Par ici, s'il vous plaît. |
| ... | ... | ... | ... |
| 6338 | 7346621793630801184 | va.plus | Guerre en Ukraine, Emmanuel Macron jouent-il b... |
| 6877 | 7389573030638374176 | franceinfo | Le 15 de France de rugby dans la tourmente. 2 ... |
| 6782 | 7382107688542440737 | franceinfo | Toi qui me regardes, à ce qu'il paraît, tu veu... |
| 6959 | 7381892076188765473 | liberation.fr | Des euh. Candidats du Rassemblement National q... |
| 6963 | 7379307364920462624 | liberation.fr | Libération, ça fait partie de son histoire. Et... |

309 rows × 3 columns


### 1.1.2 load transcripts collected using whisper

Whisper transcripts are collected by `python -m scripts.validate_whisper_transcripts` (run it first); this section only loads them.

In [12]:
tiktok_w_transcripts = pd.read_json("data/test_transcript_collection_accuracy/tiktok_test.jsonl", lines=True)

In [14]:
tiktok_w_transcripts['transcript_from_whisper'] = tiktok_w_transcripts['video_id'].apply(load_transcript)

In [15]:
tiktok_w_transcripts = tiktok_w_transcripts.dropna(subset='transcript_from_whisper')
tiktok_w_transcripts = tiktok_w_transcripts.dropna(subset='transcript')
tiktok_w_transcripts['len_transcript_1'] = tiktok_w_transcripts['transcript'].apply(len)
tiktok_w_transcripts['len_transcript_2'] = tiktok_w_transcripts['transcript_from_whisper'].apply(len)
tiktok_w_transcripts_filter = tiktok_w_transcripts[(tiktok_w_transcripts['len_transcript_1'] > 50) & (tiktok_w_transcripts['len_transcript_2'] > 50)]


## 1.2 YouTube

### 1.2.1 Create test file with video_id for which I already have the transcripts from youtube-transcripts-api

```python
# 1. Create a DF which contains 2 samples of youtube w transcripts from each channel and each file
youtube_w_transcripts = []
for year in ["2022", "2024"]:
    for channel_type in ["pp", "news"]:
        df = load_transcripts_only_df(year=year, channel_type=channel_type, platform="youtube")
        df = pd.DataFrame(df)
        #df['duration'] = df['duration'].apply(int)
        df = df[(~df['transcript'].isna()) & (df['transcript']!= '')]
        
        channels = df.channel.unique()
        
        print(f"{channel_type}, {year}, {len(channels)}")
        for channel in channels:
            df_channel = df[df['channel']==channel]
            df_channel['len_transcript'] = df_channel['transcript'].apply(len)
            df_channel = df_channel[(df_channel['len_transcript'] > 100) & (df_channel['len_transcript'] < 20000)]
            size_sample = min(2, len(df_channel))
            youtube_w_transcripts.append(df_channel.sample(size_sample, random_state=0))
youtube_w_transcripts = pd.concat(youtube_w_transcripts)
youtube_w_transcripts.to_json("data/test_transcript_collection_accuracy/youtube_test.jsonl", lines=True, orient="records")


### 1.2.2 load youtube transcripts collected using whisper

Whisper transcripts are collected by `python -m scripts.validate_whisper_transcripts` (run it first); this section only loads them.

In [16]:
youtube_w_transcripts = pd.read_json("data/test_transcript_collection_accuracy/youtube_test.jsonl", lines=True)
print(len(youtube_w_transcripts))


207


In [18]:
youtube_w_transcripts['transcript_from_whisper'] = youtube_w_transcripts['video_id'].apply(load_transcript)

In [19]:
youtube_w_transcripts = youtube_w_transcripts.dropna(subset='transcript_from_whisper')
youtube_w_transcripts = youtube_w_transcripts.dropna(subset='transcript')
youtube_w_transcripts['len_transcript_1'] = youtube_w_transcripts['transcript'].apply(len)
youtube_w_transcripts['len_transcript_2'] = youtube_w_transcripts['transcript_from_whisper'].apply(len)
youtube_w_transcripts_filter = youtube_w_transcripts[(youtube_w_transcripts['len_transcript_1'] > 50) & (youtube_w_transcripts['len_transcript_2'] > 50)]


# Add distances for pairs of transcripts 

In [20]:
pairs_df = pd.concat([youtube_w_transcripts_filter, tiktok_w_transcripts_filter]).drop(columns=['len_transcript']).rename(columns={
    'transcript':'transcript1', 'transcript_from_whisper':'transcript2', 'video_id':'videoId1'})
pairs_df['videoId2'] = pairs_df['videoId1']

In [21]:
args_dict = {'cut_transcripts':2000,
            'strip_accents':1,
            'lowercase':1,
            'remove_brackets':1,
            'remove_openers':0}

In [22]:
pairs_df_w_d, _ = add_distances(pairs_df, args_dict, preprocessing_summary = '')

# Classify pairs

In [23]:
with open('data/model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('data/best_thr.txt') as f:
    best_thr = float(f.read())
pprint.pp(get_model_coeffs(model))
print(best_thr)

{'partialratio_cut_t1': 0.19046583676153717,
 'partialratio_cut_t2': 0.13159226421984305,
 'sortratio_cut_t1': -0.009319824971382803,
 'sortratio_cut_t2': 0.027679685960971503,
 'bias': -19.459660018321404}
0.27659622233343084


/Users/Caroline/PythonEnvs/CSS/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [24]:
pairs_df_classified = classify_pairs(pairs_df_w_d, model, best_thr)
pairs_df_classified = pairs_df_classified.merge(pairs_df[['videoId1', 'transcript1', 'transcript2']])

In [25]:
from langdetect import detect

In [26]:
def is_french(text):
    if detect(text) == 'fr':return True
    else : return False

In [27]:
pairs_df_classified['is_french'] = pairs_df_classified['transcript1'].apply(is_french)
pairs_df_classified['is_french2'] = pairs_df_classified['transcript2'].apply(is_french)

In [28]:
pairs_df_classified['both_french'] = pairs_df_classified['is_french'] & pairs_df_classified['is_french2']

In [29]:
len(pairs_df_classified[~pairs_df_classified['both_french']])

10

In [30]:
pairs_df_classified_fr = pairs_df_classified[pairs_df_classified['both_french']]

In [31]:
sum(pairs_df_classified_fr['predicted_label']) / len(pairs_df_classified_fr)

0.9978586723768736

In [32]:
pairs_df_classified_fr[pairs_df_classified_fr['predicted_label'] == 0].values


array([[7093468998129421574, 7093468998129421574, 0.18935176288517097, 0,
        53, 53, 51, 51,
        '1. Dévissez les boulons de la roue. Utilisez une douille n° 16. Utilisez une clé à cliquet. 2. Dévissez les boulons de la roue. 3. Dévissez les boulons de la roue. 4. Dévissez les boulons de la roue. 5. Dévissez les boulons de la roue. 6. Dévissez les boulons de la roue.',
        "1. Dévissez les fixations du support de la douille de frein. Servez-vous d'un tournevis Phillips. 2. Dévissez les fixations du support de la douille de frein. Servez-vous d'un tournevis Phillips. 3. Retirez les fixations du support de la douille de frein.",
        True, True, True]], dtype=object)

All the transcripts that were both originally in french were correctly classified as coming from the same video, except this whisper hallucination, which is present both in the tiktok api voice_to_text and in the whisper result. 

In [39]:
np.std([fw.ratio(t1, t2) for t1, t2 in zip(pairs_df_classified_fr['transcript1'], pairs_df_classified_fr['transcript2'])])

np.float64(7.534617183254796)